In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
# Load the trained model, scaler, pickle, onehot
model=load_model('model.keras')

import pickle
with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('label_gender.pkl', 'rb') as file:
    label_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

C:\Users\rajsi\anaconda3\envs\tf_env\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(store)


In [3]:
onehot_encoder_geo.categories_

[array(['France', 'Germany', 'Spain'], dtype=object)]

In [4]:
# Example Input Data-
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary': 50000
}

In [5]:
# One-hot encode 'Geography'
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

C:\Users\rajsi\anaconda3\envs\tf_env\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [6]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [7]:
# Encode categorical variable-
input_df['Gender']=label_gender.transform(input_df['Gender'])

In [8]:
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [9]:
# Concatenate one hot encoded-
input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)

In [10]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [11]:
# Scaling the input data-
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.53223754,  0.90179633,  0.08899847, -0.67334137, -0.26522495,
         0.83017496,  0.65543311,  0.9426421 , -0.87119587,  0.98019606,
        -0.56118125, -0.57812007]])

In [12]:
# predict churn-
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step


array([[0.0255391]], dtype=float32)

In [13]:
prediction_proba = prediction[0][0]

In [14]:
prediction_proba

np.float32(0.025539104)

In [15]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
